# Phase 2 — Train Transformer dự đoán Cluster-SID

Notebook này train `EncoderDecoderRetrievalModel` trong `train_decoder.py`. Tên file source là decoder, nhưng mô hình được train ở đây chính là T5 Encoder–Decoder Transformer cho sequential recommendation.

Luồng dữ liệu:

```text
prev_items → product_index → cluster SID sequence → Transformer → next cluster SID
```

Notebook dùng trực tiếp `semantic_ids.parquet` từ notebook 03; không re-encode embedding và không cần load checkpoint RQ-VAE. Loss gồm ba cross-entropy tương ứng codebook `[128, 64, 32]`. Metric ở bước này là SID/cluster-level Hit và NDCG, chưa phải item-level ranking metric.

## Trước khi chạy

1. Bật GPU trong Kaggle Notebook Settings.
2. Add Input là output `preprocessed` của notebook 01.
3. Add Input là output `vmarket_rqvae` của notebook 03.
4. Tạo Kaggle Secret `GITHUB_TOKEN` có quyền đọc repository.
5. Tạo Kaggle Secret `WANDB_API_KEY` nếu bật W&B.
6. Nếu tự động tìm sai artifact, điền trực tiếp các đường dẫn ở cell cấu hình.

## 0. Cấu hình

In [ ]:
from pathlib import Path

SESSION_ROOT = None
RQVAE_ROOT = None
OUTPUT_ROOT = None
SESSION_CACHE_ROOT = None
PRETRAINED_TRANSFORMER_CHECKPOINT = None

GITHUB_REPOSITORY_URL = "https://github.com/nam-htran/VSF-MiniApp-Ecommerce.git"
GITHUB_BRANCH = "main"
REPOSITORY_ROOT = None

CODEBOOK_SIZES = (128, 64, 32)
MAX_SEQUENCE_LENGTH = 20
ITERATIONS = 100_000
BATCH_SIZE = 256
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-2
GRADIENT_ACCUMULATE_EVERY = 1
MAX_GRAD_NORM = 1.0
WARMUP_STEPS = 10_000

SAVE_MODEL_EVERY = 10_000
EVAL_EVERY = 2_000
FULL_EVAL_EVERY = 10_000
MAX_EVAL_BATCHES = 200

T5_D_MODEL = 128
T5_NUM_HEADS = 4
T5_D_FF = 512
T5_NUM_LAYERS = 4
TOP_K_GENERATION = 10

NUM_WORKERS = 2
USE_AMP = True
MIXED_PRECISION = "fp16"
WANDB_LOGGING = True
WANDB_PROJECT = "vmarket-transformer-training"
WANDB_ENTITY = None
WANDB_RUN_NAME = "transformer-cluster-sid"
AUTO_INSTALL_DEPENDENCIES = True
FORCE_SESSION_CACHE = False
RESET_OUTPUT = False
SKIP_IF_COMPLETE = True
SEED = 2026

print("Configuration loaded.")

## 1. Cài dependency và kiểm tra GPU

In [ ]:
import importlib.metadata as metadata
import subprocess
import sys

from packaging.version import Version


requirements = {
    "gin-config": "0.5.0",
    "accelerate": "1.0.0",
    "einops": "0.8.0",
    "transformers": "4.46.0",
    "wandb": "0.19.0",
    "pyarrow": "16.0.0",
}
packages_to_install = []
for distribution, minimum_version in requirements.items():
    try:
        installed_version = metadata.version(distribution)
    except metadata.PackageNotFoundError:
        installed_version = None
    if installed_version is None or Version(installed_version) < Version(minimum_version):
        packages_to_install.append(f"{distribution}>={minimum_version}")

if AUTO_INSTALL_DEPENDENCIES and packages_to_install:
    print("Installing:", packages_to_install)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *packages_to_install])

import json
import numpy as np
import pandas as pd
import pyarrow.compute as pc
import pyarrow.parquet as pq
import torch

if Version(torch.__version__.split("+")[0]) < Version("2.5.0"):
    raise RuntimeError(f"PyTorch >= 2.5.0 is required, found {torch.__version__}")

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", metadata.version("transformers"))
print("CUDA available:", torch.cuda.is_available())
for index in range(torch.cuda.device_count()):
    properties = torch.cuda.get_device_properties(index)
    print(f"cuda:{index}: {properties.name}, {properties.total_memory / 2**30:.1f} GiB")
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before training the Transformer.")

## 2. Kết nối Weights & Biases

Đọc `WANDB_API_KEY` từ Kaggle Secret và đăng nhập W&B.

In [ ]:
import os


if WANDB_LOGGING:
    from kaggle_secrets import UserSecretsClient

    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")

## 3. Clone source từ GitHub

Đọc `GITHUB_TOKEN` từ Kaggle Secret và clone nhánh `main`.

In [ ]:
import os

from kaggle_secrets import UserSecretsClient


github_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
if REPOSITORY_ROOT is None:
    REPOSITORY_ROOT = Path("/kaggle/working/vsf-miniapp-ecommerce-source")
REPOSITORY_ROOT = Path(REPOSITORY_ROOT).expanduser().resolve()

git_environment = {
    **os.environ,
    "GITHUB_TOKEN": github_token,
    "GIT_TERMINAL_PROMPT": "0",
}
credential_helper = "!f() { echo username=x-access-token; echo password=$GITHUB_TOKEN; }; f"
git = ["git", "-c", f"credential.helper={credential_helper}"]

if (REPOSITORY_ROOT / ".git").is_dir():
    subprocess.run(
        [*git, "-C", str(REPOSITORY_ROOT), "pull", "--ff-only", "origin", GITHUB_BRANCH],
        check=True,
        env=git_environment,
    )
elif REPOSITORY_ROOT.exists():
    raise FileExistsError(f"Clone target is not a Git repository: {REPOSITORY_ROOT}")
else:
    subprocess.run(
        [*git, "clone", "--depth", "1", "--branch", GITHUB_BRANCH, GITHUB_REPOSITORY_URL, str(REPOSITORY_ROOT)],
        check=True,
        env=git_environment,
    )
del github_token, git_environment

SOURCE_ROOT = REPOSITORY_ROOT / "ai-recommendation/src"
if not (SOURCE_ROOT / "train_decoder.py").is_file():
    raise FileNotFoundError(f"Transformer source not found: {SOURCE_ROOT}")
print("SOURCE_ROOT:", SOURCE_ROOT)

## 4. Tìm source và input artifacts

In [ ]:
def find_input_root(filename):
    matches = list(Path("/kaggle/input").glob(f"**/{filename}"))
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected one {filename}, found {len(matches)}.")
    return matches[0].parent.resolve()


SESSION_ROOT = find_input_root("model_sessions_train.parquet") if SESSION_ROOT is None else Path(SESSION_ROOT).expanduser().resolve()
RQVAE_ROOT = find_input_root("semantic_ids.parquet") if RQVAE_ROOT is None else Path(RQVAE_ROOT).expanduser().resolve()
if OUTPUT_ROOT is None:
    OUTPUT_ROOT = Path("/kaggle/working/vmarket_transformer")
if SESSION_CACHE_ROOT is None:
    SESSION_CACHE_ROOT = Path("/kaggle/working/vmarket_transformer_session_cache")
OUTPUT_ROOT = Path(OUTPUT_ROOT).expanduser().resolve()
SESSION_CACHE_ROOT = Path(SESSION_CACHE_ROOT).expanduser().resolve()

print("SESSION_ROOT:", SESSION_ROOT)
print("RQVAE_ROOT:", RQVAE_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("SESSION_CACHE_ROOT:", SESSION_CACHE_ROOT)

## 5. Kiểm tra session và Semantic ID contract

Lịch sử dài hơn 20 sản phẩm sẽ giữ lại 20 sản phẩm gần nhất. Với Amazon-M2 hiện tại, percentile 99 của độ dài lịch sử vào khoảng 18 nên mức cắt này chỉ ảnh hưởng phần đuôi rất dài.

In [ ]:
session_summary = {}
for split in ["train", "validation"]:
    path = SESSION_ROOT / f"model_sessions_{split}.parquet"
    parquet_file = pq.ParquetFile(path)
    if parquet_file.schema_arrow.names != ["prev_items", "next_item"]:
        raise ValueError(f"Unexpected {split} session schema: {parquet_file.schema_arrow.names}")
    histories = pq.read_table(path, columns=["prev_items"])["prev_items"]
    lengths = pc.list_value_length(histories).combine_chunks()
    quantiles = pc.quantile(
        lengths,
        q=[0.5, 0.9, 0.95, 0.99, 1.0],
        interpolation="nearest",
    ).to_pylist()
    session_summary[split] = {
        "rows": parquet_file.metadata.num_rows,
        "mean_length": float(pc.mean(lengths).as_py()),
        "p50_p90_p95_p99_max": quantiles,
    }

semantic_ids_path = RQVAE_ROOT / "semantic_ids.parquet"
semantic_file = pq.ParquetFile(semantic_ids_path)
expected_columns = ["product_index", "product_id", "sid_0", "sid_1", "sid_2"]
if semantic_file.schema_arrow.names != expected_columns:
    raise ValueError(f"Unexpected semantic ID schema: {semantic_file.schema_arrow.names}")
sid_table = pq.read_table(semantic_ids_path, columns=["sid_0", "sid_1", "sid_2"])
sid_ranges = {}
for layer, codebook_size in enumerate(CODEBOOK_SIZES):
    values = sid_table.column(f"sid_{layer}").combine_chunks().to_numpy()
    minimum, maximum = int(values.min()), int(values.max())
    if minimum < 0 or maximum >= codebook_size:
        raise ValueError(f"sid_{layer} outside [0, {codebook_size - 1}]")
    sid_ranges[f"sid_{layer}"] = [minimum, maximum]

print("Input validation: PASSED")
print(json.dumps(session_summary, indent=2))
print("Semantic ID rows:", f"{semantic_file.metadata.num_rows:,}")
print("SID ranges:", sid_ranges)

## 6. Tạo Gin config

In [ ]:
import re
import shutil
if RESET_OUTPUT and OUTPUT_ROOT.exists():
    if OUTPUT_ROOT.name != "vmarket_transformer":
        raise ValueError(f"Refusing to reset an unexpected output path: {OUTPUT_ROOT}")
    shutil.rmtree(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SESSION_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

base_config_path = SOURCE_ROOT / "configs/transformer_vmarket.gin"
runtime_config_path = SOURCE_ROOT / "configs/transformer_kaggle_runtime.gin"
config_text = base_config_path.read_text(encoding="utf-8")
overrides = {
    "train.iterations": str(ITERATIONS),
    "train.batch_size": str(BATCH_SIZE),
    "train.learning_rate": repr(LEARNING_RATE),
    "train.weight_decay": repr(WEIGHT_DECAY),
    "train.session_root": json.dumps(str(SESSION_ROOT)),
    "train.semantic_ids_path": json.dumps(str(semantic_ids_path)),
    "train.session_cache_root": json.dumps(str(SESSION_CACHE_ROOT)),
    "train.save_dir_root": json.dumps(str(OUTPUT_ROOT)),
    "train.codebook_sizes": json.dumps(list(CODEBOOK_SIZES)),
    "train.max_sequence_length": str(MAX_SEQUENCE_LENGTH),
    "train.force_session_cache": str(FORCE_SESSION_CACHE),
    "train.num_workers": str(NUM_WORKERS),
    "train.amp": str(USE_AMP),
    "train.mixed_precision_type": json.dumps(MIXED_PRECISION),
    "train.gradient_accumulate_every": str(GRADIENT_ACCUMULATE_EVERY),
    "train.max_grad_norm": repr(MAX_GRAD_NORM),
    "train.warmup_steps": str(WARMUP_STEPS),
    "train.save_model_every": str(SAVE_MODEL_EVERY),
    "train.eval_every": str(EVAL_EVERY),
    "train.full_eval_every": str(FULL_EVAL_EVERY),
    "train.max_eval_batches": str(MAX_EVAL_BATCHES),
    "train.wandb_logging": str(WANDB_LOGGING),
    "train.wandb_project": json.dumps(WANDB_PROJECT),
    "train.wandb_entity": "None" if WANDB_ENTITY is None else json.dumps(WANDB_ENTITY),
    "train.wandb_run_name": "None" if WANDB_RUN_NAME is None else json.dumps(WANDB_RUN_NAME),
    "train.t5_d_model": str(T5_D_MODEL),
    "train.t5_num_heads": str(T5_NUM_HEADS),
    "train.t5_d_ff": str(T5_D_FF),
    "train.t5_num_layers": str(T5_NUM_LAYERS),
    "train.top_k_for_generation": str(TOP_K_GENERATION),
    "train.should_add_sep_token": "True",
    "train.top_k_eval_list": "[1, 5, 10]",
    "train.seed": str(SEED),
}
for key, value in overrides.items():
    pattern = rf"(?m)^{re.escape(key)}=.*$"
    config_text, replacement_count = re.subn(pattern, f"{key}={value}", config_text)
    if replacement_count != 1:
        raise ValueError(f"Expected exactly one Gin binding for {key}, found {replacement_count}")

if PRETRAINED_TRANSFORMER_CHECKPOINT is not None:
    checkpoint_path = Path(PRETRAINED_TRANSFORMER_CHECKPOINT).expanduser().resolve()
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f"Transformer checkpoint not found: {checkpoint_path}")
    config_text += f"\ntrain.pretrained_decoder_path={json.dumps(str(checkpoint_path))}\n"

runtime_config_path.write_text(config_text, encoding="utf-8")
print("Runtime Gin config:", runtime_config_path)
print("\n" + config_text)

## 7. Train Transformer

Lần chạy đầu sẽ tạo cache session dạng NumPy cố định trong `/kaggle/working`; bước này ánh xạ khoảng 3,6 triệu session từ product ID sang product index và có thể mất vài phút. Các lần chạy sau sẽ tái sử dụng cache nếu contract không đổi.

In [ ]:
import os


metrics_path = OUTPUT_ROOT / "transformer_metrics.json"
if SKIP_IF_COMPLETE and metrics_path.is_file():
    print("Complete Transformer artifacts already exist; skipping training.")
else:
    command = [sys.executable, "train_decoder.py", str(runtime_config_path)]
    environment = os.environ.copy()
    environment["PYTHONUNBUFFERED"] = "1"
    environment["WANDB_SILENT"] = "true"
    print("Running:", " ".join(command))
    process = subprocess.Popen(
        command,
        cwd=SOURCE_ROOT,
        env=environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f"Transformer training failed with exit code {return_code}")

print("Training cell finished.")

## 8. Kiểm tra output

In [ ]:
if not metrics_path.is_file():
    raise FileNotFoundError(f"Transformer metrics not found: {metrics_path}")
checkpoints = sorted(OUTPUT_ROOT.glob("checkpoint_*.pt"))
if not checkpoints:
    raise FileNotFoundError("No Transformer checkpoint was written.")
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
cache_manifest_path = SESSION_CACHE_ROOT / "session_cache_manifest.json"
if not cache_manifest_path.is_file():
    raise FileNotFoundError("Session cache manifest was not written.")
cache_manifest = json.loads(cache_manifest_path.read_text(encoding="utf-8"))
output_size = sum(path.stat().st_size for path in OUTPUT_ROOT.rglob("*") if path.is_file())

print("Final Transformer validation: PASSED")
print("Latest checkpoint:", checkpoints[-1])
print("Output size:", f"{output_size / 2**30:.2f} GiB")
print("Truncated sessions:", cache_manifest["truncated_sessions"])
display(pd.DataFrame([metrics]).T.rename(columns={0: "value"}))
print("OUTPUT_ROOT:", OUTPUT_ROOT)

## Kết quả của bước này

Notebook hoàn thành khi cell cuối báo `Final Transformer validation: PASSED`. Output là Transformer có thể sinh Top-K cluster SID. Bước tiếp theo mới mở rộng cluster thành candidate item và đánh giá/ranking ở item level.